In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
import torch.optim as optim
import pandas as pd

In [ ]:
data = pd.read_csv('../data/train.csv')

In [ ]:
data = data[['pIC50','Smiles']]

In [ ]:
data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("seyonec/PubChem10M_SMILES_BPE_450k")
base_model = AutoModel.from_pretrained("seyonec/PubChem10M_SMILES_BPE_450k")

In [ ]:
class SMILESRegressor(nn.Module):
    def __init__(self, base_model):
        super(SMILESRegressor, self).__init__()
        self.base_model = base_model
        self.regression_head = nn.Sequential(
            nn.Linear(base_model.config.hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)  # pIC50 예측을 위한 단일 출력
        )

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # CLS 토큰 벡터 사용
        return self.regression_head(hidden_state)

In [ ]:
model = SMILESRegressor(base_model)

In [ ]:
smiles_data = data['Smiles'].tolist()
labels = torch.tensor(data['pIC50']).unsqueeze(1)  # 레이블 데이터

In [ ]:
inputs = tokenizer(smiles_data, padding=True, truncation=True, return_tensors="pt")

input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

In [ ]:
from tqdm import tqdm

In [ ]:
num_epochs = 10  # 원하는 에폭 수 설정
model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    with tqdm(total=len(input_ids), desc=f"Epoch {epoch+1}/{num_epochs}", unit="sample") as pbar:
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({"Loss": loss.item()})
        pbar.update(input_ids.size(0))
    
    print(f"Epoch {epoch+1}/{num_epochs} completed. Average Loss: {epoch_loss/len(input_ids)}")

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(input_ids=input_ids, attention_mask=attention_mask)
    print(predictions)